<a href="https://colab.research.google.com/github/sachinsitapure/HF_Chatbot/blob/main/HF_ChatBoT_FINAL.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install transformers torch accelerate

In [7]:
!pip install --upgrade huggingface_hub
# Alternatively, to ensure CLI tools are included
#!pip install -U "huggingface_hub[cli]"


In [10]:
!huggingface-cli login
!huggingface-cli whoami

/bin/bash: line 1: huggingface-cli: command not found
/bin/bash: line 1: huggingface-cli: command not found


Loading Model & Tokenizer
Here, we are preparing our session by loading both the Llama model and its associated tokenizer.

The tokenizer will help in converting our text prompts into a format that the model can understand and process.

In [12]:
from transformers import AutoTokenizer
import transformers
import torch
from huggingface_hub import login

# Log in to Hugging Face to access gated models
login()

model = "meta-llama/Llama-2-7b-chat-hf" # meta-llama/Llama-2-7b-hf

tokenizer = AutoTokenizer.from_pretrained(model, token=True)

config.json:   0%|          | 0.00/614 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.62k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.84M [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/414 [00:00<?, ?B/s]

Creating the Llama Pipeline
We'll set up a pipeline for text generation.

This pipeline simplifies the process of feeding prompts to our model and receiving generated text as output.

Note: This cell takes 2-3 minutes to run

In [13]:
from transformers import pipeline

llama_pipeline = pipeline(
    "text-generation",  # LLM task
    model=model,
    torch_dtype=torch.float16,
    device_map="auto",
)

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json:   0%|          | 0.00/26.8k [00:00<?, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/188 [00:00<?, ?B/s]

#Getting Responses
With everything set up, let's see how Llama responds to some sample queries.

In [ ]:
def get_llama_response(prompt: str) -> None:
    """
    Generate a response from the Llama model.

    Parameters:
        prompt (str): The user's input/question for the model.

    Returns:
        None: Prints the model's response.
    """
    sequences = llama_pipeline(
        prompt,
        do_sample=True,
        top_k=10,
        num_return_sequences=1,
        eos_token_id=tokenizer.eos_token_id,
        max_length=256,
    )
    #print("Chatbot:", sequences[0]['generated_text'])
    return sequences[0]['generated_text']



prompt = 'I liked "Breaking Bad" and "Band of Brothers". Do you have any recommendations of other shows I might like?\n'
#get_llama_response(prompt)

In [16]:
def chatbot():
    messages = [
        {"role": "system", "content": "You are a helpful assistant."}
    ]

    while True:
        user_input = input("User: ")

        if user_input.lower() in ["quit", "bye", "exit"]:
            print("Goodbye...")
            break

        messages.append({"role": "user", "content": user_input})

        response = get_llama_response(messages)

        print("Bot:", response)

        messages.append({"role": "assistant", "content": response})


if __name__ == "__main__":
    print("Start chatting with the bot (type 'quit' to stop)!")
    chatbot()

Start chatting with the bot (type 'quit' to stop)!
User: What is difference between cricket and football?
Bot: Cricket and football are two popular sports with distinct rules, equipment, and gameplay. Here's a brief overview of the main differences:

1. **Origin and Global Popularity:**
   - Cricket originated in England in the 16th century, with a significant following in the UK, Australia, India, Pakistan, and other Commonwealth countries.
   - Football (soccer) originated in Britain in the mid-19th century, with a massive global following, especially in Europe, South America, Africa, and Asia.

2. **Equipment:**
   - Cricket: Played with a flat, hard bat, a hard ball, and protective gear like helmets, pads, and gloves.
   - Football: Played with a round ball and a player's feet, legs, and head.

3. **Number of Players:**
   - Cricket: Two teams with 11 players each (including a captain) on the field at a time.
   - Football: Two teams with 11 players each on the field at a time, wit

In [14]:
from huggingface_hub import InferenceClient

client = InferenceClient()

def get_llama_response(messages):
    response = client.chat.completions.create(
        model="meta-llama/Meta-Llama-3-8B-Instruct",
        messages=messages,
        max_tokens=256
    )
    return response.choices[0].message.content


def chatbot():
    messages = [
        {"role": "system", "content": "You are a helpful assistant."}
    ]

    while True:
        user_input = input("User: ")

        if user_input.lower() in ["quit", "bye", "exit"]:
            print("Goodbye...")
            break

        messages.append({"role": "user", "content": user_input})

        response = get_llama_response(messages)

        print("Bot:", response)

        messages.append({"role": "assistant", "content": response})


if __name__ == "__main__":
    print("Start chatting with the bot (type 'quit' to stop)!")
    chatbot()


Start chatting with the bot (type 'quit' to stop)!
User: What is machine learning?
Bot: Machine learning is a subset of artificial intelligence (AI) that involves the use of algorithms and statistical models to enable machines to learn from data, make decisions, and improve their performance over time. The primary goal of machine learning is to enable computers to learn from experience, rather than being explicitly programmed for a specific task.

Machine learning algorithms can be classified into three main categories:

1. **Supervised learning**: The algorithm is trained on labeled data, where the correct output is already known. The algorithm learns to map inputs to outputs based on the labeled data, and can then make predictions on new, unseen data.
2. **Unsupervised learning**: The algorithm is trained on unlabeled data, and must find patterns or structure in the data on its own. This approach is often used for clustering, dimensionality reduction, and anomaly detection.
3. **Rein

In [ ]:
def main():
    print("Start chatting with the bot (type 'quit' to stop)!")
    chatbot()